In [ ]:
import os
import sys
import time
import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

# 路径设置（假设 notebook 在 tagging/pure_unsmear/seperated 下运行）。
THIS_DIR = os.getcwd()
MODULE_DIR = THIS_DIR
TAGGING_DIR = os.path.abspath(os.path.join(MODULE_DIR, '..'))

sys.path.insert(0, MODULE_DIR)

import tool  # noqa: E402
from model import TokenMLP, TokenTransformerRegressor, CondFlowMatcher  # noqa: E402

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

RUN_NAME = 'pure_unsmear_axisown_mlp_transformer_fm'
OUT_DIR = os.path.join(MODULE_DIR, 'runs', RUN_NAME)
FIG_DIR = os.path.join(OUT_DIR, 'figs')
CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
SHARED_BASELINE_DIR = os.path.join(MODULE_DIR, 'runs', 'shared_offline_hlt_baselines')
SHARED_BASELINE_CKPT_DIR = os.path.join(SHARED_BASELINE_DIR, 'ckpts')
EXTERNAL_BASELINE_RUN_DIR = os.path.join(TAGGING_DIR, 'Baseline_smearing', 'runs', 'smear_only_offline_hlt_hltkd_repeat3')
EXTERNAL_BASELINE_REPEAT_DIR = os.path.join(EXTERNAL_BASELINE_RUN_DIR, 'repeats')

tool.ensure_dir(FIG_DIR)
tool.ensure_dir(CKPT_DIR)
tool.ensure_dir(SHARED_BASELINE_CKPT_DIR)

CONFIG = {
    'data_path': os.path.join(TAGGING_DIR, 'test.h5'),
    'load_shared_baselines': True,
    'load_unsmear_bundle': False,
    'n_jets': 200000,
    'max_particles': 100,
    'downstream_repeat_seeds': [42, 52, 62],
    'kd_teacher_repeat_seed': 62,
    'external_baseline_repeat_dir': EXTERNAL_BASELINE_REPEAT_DIR,
    # Unsmear training features: '3d'/'4d'/'7d'
    'feature_kind': '7d',
    # Use 7d uniformly for downstream tagger AUC comparisons
    'downstream_feature_kind': '7d',
    # Source of the last 3 dimensions in downstream 7d: 'direct' or 'rebuild_from_first4'
    'downstream_post3_source': 'direct',
    'consistency': {
        'w_dr': 0.0,
    },
    'uncertainty': {
        'enabled': False,
        'on_features': ['log_E', 'log_pt_rel', 'log_E_rel', 'dR'],
        'log_var_clip': 6.0,
    },
    'hlt_effects': {
        # Pure unsmear: use the same threshold=0.5 for offline / HLT and keep only smearing effects
        'pt_threshold_offline': 0,
        'pt_threshold_hlt': 0,
        'pt_resolution': 0.10,
        'eta_resolution': 0.03,
        'phi_resolution': 0.03,
    },
    # 下游 tagger 结构配置。
    'tagger': {
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 6,
        'ff_dim': 512,
        'dropout': 0.1,
    },
    # 下游 tagger 训练超参。
    'tagger_training': {
        'batch_size': 512,
        'epochs': 50,
        'lr': 5e-4,
        'weight_decay': 1e-5,
        'warmup_epochs': 3,
        'patience': 8,
    },
    # 下游 KD 权重与温度。
    'tagger_kd': {
        'temperature': 3.0,
        'alpha_kd': 0.5,
        'alpha_attn': 0.0,
    },
    'mlp': {
        # input_dim will be overwritten below according to feature_kind
        'input_dim': 7,
        'hidden_dim': 256,
        'num_layers': 4,
        'dropout': 0.1,
        'return_reco': True,
        'predict_logvar': False,
        # Mask option: if False, it degenerates to using the mask only in the loss
        'add_mask_channel': False,
        'mask_output': True,
    },
    'transformer': {
        # input_dim will be overwritten below according to feature_kind
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 6,
        'ff_dim': 512,
        'dropout': 0.1,
        'return_reco': True,
        'predict_logvar': False,
        'add_mask_channel': False,
        'mask_output': True,
        'use_positional_embedding': False,
        'max_seq_len': 128,
    },
    'fm': {
        # input_dim will be overwritten below according to feature_kind
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 4,
        'ff_dim': 512,
        'dropout': 0.1,
        'time_n_freqs': 16,
        'time_max_freq': 200.0,
        'time_t_eps': 1e-4,
        'per_layer_cond': True,
        'sampler': 'heun',
        'sample_steps': 30,
    },
    'training': {
        'batch_size': 256,
        'epochs': 40,
        'lr': 5e-4,
        'weight_decay': 1e-5,
        'warmup_epochs': 3,
        'patience': 6,
        'resmear_each_epoch': True,
        'resmear_seed_stride': 1,
    },
}

# 自动对齐特征维度。
feat_names = tool.get_feat_names(CONFIG['feature_kind'])
D = len(feat_names)
CONFIG['mlp']['input_dim'] = D
CONFIG['transformer']['input_dim'] = D
CONFIG['transformer']['max_seq_len'] = int(CONFIG['max_particles'])
CONFIG['fm']['input_dim'] = D
CONFIG['mlp']['predict_logvar'] = bool(CONFIG['uncertainty']['enabled'])
CONFIG['transformer']['predict_logvar'] = bool(CONFIG['uncertainty']['enabled'])

# 下游 tagger 的输入维度由 downstream_feature_kind 决定。
down_feat_names = tool.get_feat_names(CONFIG['downstream_feature_kind'])
CONFIG['tagger']['input_dim'] = len(down_feat_names)

print('Data path:', CONFIG['data_path'])
print('Run dir:', OUT_DIR)
print('Feature kind:', CONFIG['feature_kind'], 'input_dim:', D, 'feat_names:', feat_names)
print('Downstream feature kind:', CONFIG['downstream_feature_kind'], 'input_dim:', CONFIG['tagger']['input_dim'])
print('Repeat seeds:', CONFIG['downstream_repeat_seeds'])
print('KD teacher repeat seed:', CONFIG['kd_teacher_repeat_seed'])
print('External baseline repeat dir:', CONFIG['external_baseline_repeat_dir'])
print('Tagger cfg:', CONFIG['tagger'])
print('Tagger train cfg:', CONFIG['tagger_training'])
print('Tagger KD cfg:', CONFIG['tagger_kd'])


In [2]:
# Load raw constituents
n = int(CONFIG['n_jets'])
S = int(CONFIG['max_particles'])

with h5py.File(CONFIG['data_path'], 'r') as f:
    labels = f['labels'][:n].astype(np.int64)
    weights = f['weights'][:n].astype(np.float32)
    pt = f['fjet_clus_pt'][:n, :S].astype(np.float32)
    eta = f['fjet_clus_eta'][:n, :S].astype(np.float32)
    phi = f['fjet_clus_phi'][:n, :S].astype(np.float32)
    E = f['fjet_clus_E'][:n, :S].astype(np.float32)

constituents_raw = np.stack([pt, eta, phi, E], axis=-1)  # [N,S,4]
mask_raw = pt > 0

print('Raw:', constituents_raw.shape, 'mask:', mask_raw.shape)
print('Signal:', int(labels.sum()), 'Bkg:', int((1 - labels).sum()))


Raw: (200000, 100, 4) mask: (200000, 100)
Signal: 99836 Bkg: 100164


In [3]:
# Build supervision pairs: target=unsmeared view, input=smeared HLT view
hcfg = tool.HLTEffectsCfg(**CONFIG['hlt_effects'])

pre_const, post_const, post_mask = tool.apply_hlt_effects_pair(
    constituents_raw,
    mask_raw,
    hcfg,
    seed=seed,
) 
# Note: HLT E depends on pt and eta.

pre_const = pre_const.copy()
post_const = post_const.copy()
pre_const[~post_mask] = 0.0
post_const[~post_mask] = 0.0
assert np.all(pre_const[~post_mask] == 0.0) and np.all(post_const[~post_mask] == 0.0)

print('Pre/post:', pre_const.shape, post_const.shape, 'mask:', post_mask.shape)
print('Avg tokens per jet (post):', float(post_mask.sum(axis=1).mean()))

axis_pre = tool.compute_jet_axis(pre_const, post_mask)
axis_post = tool.compute_jet_axis(post_const, post_mask)
feat_pre = tool.compute_features_with_axis(pre_const, post_mask, axis_pre, kind=CONFIG['feature_kind'])  # Use post_mask here because the current thresholds are identical, so the same post_mask can be shared.
feat_post = tool.compute_features_with_axis(post_const, post_mask, axis_post, kind=CONFIG['feature_kind'])

print('Features:', feat_pre.shape, feat_post.shape, 'kind:', CONFIG['feature_kind'])

# split (jet-level)
idx = np.arange(len(labels))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=seed, stratify=labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=seed, stratify=labels[temp_idx])
print(f"Split: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}")


Pre/post: (200000, 100, 4) (200000, 100, 4) mask: (200000, 100)
Avg tokens per jet (post): 54.29243
Features: (200000, 100, 7) (200000, 100, 7) kind: 7d
Split: train=140,000 val=30,000 test=30,000


In [4]:
# Standardize using unsmeared-target train statistics
feat_means, feat_stds = tool.get_stats(feat_pre, post_mask, train_idx)

x_train = tool.standardize(feat_post, post_mask, feat_means, feat_stds, clip=10.0)
y_train = tool.standardize(feat_pre, post_mask, feat_means, feat_stds, clip=10.0)

print('Standardization done.')
print('Means:', np.round(feat_means, 4))
print('Stds :', np.round(feat_stds, 4))


Standardization done.
Means: [-2.0000e-04 -1.0000e-04  8.7940e+00  9.0840e+00 -5.2585e+00 -5.2701e+00
  2.2250e-01]
Stds : [0.2121 0.2173 1.5182 1.5217 1.4919 1.4935 0.2067]


In [5]:
# DataLoaders
BS = int(CONFIG['training']['batch_size'])

train_ds = tool.UnsmearJetDataset(x_train[train_idx], y_train[train_idx], post_mask[train_idx])
val_ds = tool.UnsmearJetDataset(x_train[val_idx], y_train[val_idx], post_mask[val_idx])
test_ds = tool.UnsmearJetDataset(x_train[test_idx], y_train[test_idx], post_mask[test_idx])

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BS)
test_loader = DataLoader(test_ds, batch_size=BS)

# During training, re-smear the inputs every epoch and recompute the post-side jet axis / features.
train_const_raw = constituents_raw[train_idx]
train_mask_raw = mask_raw[train_idx]


def make_epoch_train_loader(epoch: int):
    if not bool(CONFIG['training'].get('resmear_each_epoch', False)):
        return train_loader
    stride = int(CONFIG['training'].get('resmear_seed_stride', 1))
    epoch_seed = int(seed + (int(epoch) - 1) * stride)
    x_ep, y_ep, m_ep = tool.build_unsmear_epoch_arrays(
        train_const_raw,
        train_mask_raw,
        hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=epoch_seed,
        clip=10.0,
    )
    ds = tool.UnsmearJetDataset(x_ep, y_ep, m_ep)
    return DataLoader(ds, batch_size=BS, shuffle=True, drop_last=True)


print('Loaders ready, BS=', BS)
print('Epoch resmear enabled:', bool(CONFIG['training'].get('resmear_each_epoch', False)))

Loaders ready, BS= 256
Epoch resmear enabled: True


In [ ]:
# Repeat-train MLP / Transformer / FM regression vs Flow Matching
train_cfg = CONFIG['training']
epochs = int(train_cfg['epochs'])
lr = float(train_cfg['lr'])
wd = float(train_cfg['weight_decay'])
warm = int(train_cfg['warmup_epochs'])
pat = int(train_cfg['patience'])


def make_scheduler(opt):
    def lr_lambda(ep):
        if ep < warm:
            return float(ep + 1) / float(max(1, warm))
        t = (ep - warm) / float(max(1, epochs - warm))
        return 0.5 * (1.0 + np.cos(np.pi * t))

    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


# 损失相关辅助。
use_unc = bool(CONFIG['uncertainty']['enabled'])
w_dr = float(CONFIG['consistency']['w_dr'])

# 特征索引。
idx_map = {n: i for i, n in enumerate(feat_names)}
dphi_idx = idx_map.get('dPhi', None)
deta_idx = idx_map.get('dEta', None)
dr_idx = idx_map.get('dR', None)

if dphi_idx is not None:
    dphi_mean = float(feat_means[int(dphi_idx)])
    dphi_scale = float(feat_stds[int(dphi_idx)])
else:
    dphi_mean = 0.0
    dphi_scale = 1.0

active_dim_mask = None
if use_unc:
    on = set(CONFIG['uncertainty'].get('on_features', []))
    active_dim_mask = torch.tensor([name in on for name in feat_names], dtype=torch.bool)


def regression_loss(mu: torch.Tensor, y: torch.Tensor, m: torch.Tensor, *, log_var: torch.Tensor | None = None) -> torch.Tensor:
    if use_unc:
        assert log_var is not None
        clip = float(CONFIG['uncertainty'].get('log_var_clip', 6.0))
        if dphi_idx is not None:
            base = tool.masked_gaussian_nll_wrap_dphi(
                mu,
                log_var,
                y,
                m,
                dphi_idx=int(dphi_idx),
                dphi_scale=dphi_scale,
                active_dim_mask=active_dim_mask,
                log_var_clip=clip,
            )
        else:
            base = tool.masked_gaussian_nll(
                mu,
                log_var,
                y,
                m,
                active_dim_mask=active_dim_mask,
                log_var_clip=clip,
            )
    else:
        if dphi_idx is not None:
            base = tool.masked_smooth_l1_wrap_dphi(mu, y, m, dphi_idx=int(dphi_idx), dphi_scale=dphi_scale)
        else:
            base = tool.masked_smooth_l1(mu, y, m)

    if (w_dr > 0.0) and (dr_idx is not None) and (deta_idx is not None) and (dphi_idx is not None):
        dR_cons = torch.sqrt(mu[..., int(deta_idx)] ** 2 + mu[..., int(dphi_idx)] ** 2 + 1e-12)
        dR_pred = mu[..., int(dr_idx)]
        cons = tool.masked_smooth_l1(dR_pred.unsqueeze(-1), dR_cons.unsqueeze(-1), m)
        base = base + float(w_dr) * cons

    return base


@torch.no_grad()
def eval_regression(model, loader):
    model.eval()
    tot = 0.0
    n = 0
    for batch in loader:
        x = batch['x'].to(device)
        y = batch['y'].to(device)
        m = batch['mask'].to(device)
        out = model(x, m)
        if isinstance(out, tuple):
            mu, log_var = out
        else:
            mu, log_var = out, None
        loss = regression_loss(mu, y, m, log_var=log_var)
        tot += float(loss.item()) * int(x.shape[0])
        n += int(x.shape[0])
    return tot / max(1, n)


@torch.no_grad()
def eval_fm(model, loader, *, steps: int):
    model.eval()
    tot = 0.0
    n = 0
    for batch in loader:
        x0 = batch['x'].to(device)
        y = batch['y'].to(device)
        m = batch['mask'].to(device)
        sampler = str(CONFIG['fm'].get('sampler', 'euler')).lower()
        if sampler == 'heun':
            pred = tool.fm_sample_heun(model, x0=x0, cond=x0, mask=m, steps=int(steps), dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
        else:
            pred = tool.fm_sample_euler(model, x0=x0, cond=x0, mask=m, steps=int(steps), dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
        if dphi_idx is not None:
            loss = tool.masked_smooth_l1_wrap_dphi(pred, y, m, dphi_idx=int(dphi_idx), dphi_scale=dphi_scale)
        else:
            loss = tool.masked_smooth_l1(pred, y, m)
        tot += float(loss.item()) * int(x0.shape[0])
        n += int(x0.shape[0])
    return tot / max(1, n)


with torch.no_grad():
    x_hlt_test = torch.from_numpy(x_train[test_idx]).to(device)
    y_off_test = torch.from_numpy(y_train[test_idx]).to(device)
    m_test = torch.from_numpy(post_mask[test_idx]).to(device)
    if dphi_idx is not None:
        hlt_test_loss = tool.masked_smooth_l1_wrap_dphi(
            x_hlt_test,
            y_off_test,
            m_test,
            dphi_idx=int(dphi_idx),
            dphi_scale=dphi_scale,
        )
    else:
        hlt_test_loss = tool.masked_smooth_l1(x_hlt_test, y_off_test, m_test)
    print('HLT->OFF test baseline loss (masked_smooth_l1_wrap_dphi):', float(hlt_test_loss.item()))


def _set_repeat_seed(local_seed: int):
    np.random.seed(int(local_seed))
    torch.manual_seed(int(local_seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(local_seed))


def _predict_regression_arrays(model, x_std_np: np.ndarray, mask_np: np.ndarray, *, bs: int = 512) -> np.ndarray:
    model.eval()
    ds = tool.UnsmearJetDataset(x_std_np, x_std_np, mask_np)
    ld = DataLoader(ds, batch_size=int(bs), shuffle=False)
    outs = []
    with torch.no_grad():
        for batch in ld:
            x = batch['x'].to(device)
            m = batch['mask'].to(device)
            out = model(x, m)
            mu = out[0] if isinstance(out, tuple) else out
            outs.append(mu.cpu().numpy())
    return np.concatenate(outs, axis=0)


def _predict_fm_arrays(model, x_std_np: np.ndarray, mask_np: np.ndarray, *, bs: int = 512) -> np.ndarray:
    model.eval()
    ds = tool.UnsmearJetDataset(x_std_np, x_std_np, mask_np)
    ld = DataLoader(ds, batch_size=int(bs), shuffle=False)
    outs = []
    fm_steps = int(CONFIG['fm']['sample_steps'])
    fm_sampler = str(CONFIG['fm'].get('sampler', 'euler')).lower()
    with torch.no_grad():
        for batch in ld:
            x0 = batch['x'].to(device)
            m = batch['mask'].to(device)
            if fm_sampler == 'heun':
                pred = tool.fm_sample_heun(model, x0=x0, cond=x0, mask=m, steps=fm_steps, dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
            else:
                pred = tool.fm_sample_euler(model, x0=x0, cond=x0, mask=m, steps=fm_steps, dphi_idx=dphi_idx, dphi_mean=dphi_mean, dphi_scale=dphi_scale)
            outs.append(pred.cpu().numpy())
    return np.concatenate(outs, axis=0)


def _save_rows_csv(rows: list[dict], out_path: str):
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print('Saved table:', out_path)


UNSMEAR_REPEAT_DIR = os.path.join(CKPT_DIR, 'unsmear_model_repeats')
UNSMEAR_REPEAT_TABLE_DIR = os.path.join(UNSMEAR_REPEAT_DIR, 'tables')
tool.ensure_dir(UNSMEAR_REPEAT_DIR)
tool.ensure_dir(UNSMEAR_REPEAT_TABLE_DIR)

UNSMEAR_MODEL_SPECS = [
    {
        'display_name': 'MLP',
        'short_name': 'mlp',
        'kind': 'regression',
    },
    {
        'display_name': 'Transformer',
        'short_name': 'transformer',
        'kind': 'regression',
    },
    {
        'display_name': 'FM',
        'short_name': 'fm',
        'kind': 'fm',
    },
]

REPEAT_SEEDS = [int(x) for x in CONFIG.get('downstream_repeat_seeds', [42, 52, 62])]
REPRESENTATIVE_UNSMEAR_REPEAT_SEED = int(REPEAT_SEEDS[0])


def _make_unsmear_model(spec: dict):
    if spec['short_name'] == 'mlp':
        return TokenMLP(**CONFIG['mlp']).to(device)
    if spec['short_name'] == 'transformer':
        return TokenTransformerRegressor(**CONFIG['transformer']).to(device)
    if spec['short_name'] == 'fm':
        fm_cfg = dict(CONFIG['fm'])
        _ = fm_cfg.pop('sample_steps')
        _ = fm_cfg.pop('sampler', None)
        return CondFlowMatcher(**fm_cfg).to(device)
    raise KeyError(spec['short_name'])


def _unsmear_repeat_dir(repeat_idx: int, repeat_seed: int) -> str:
    return os.path.join(UNSMEAR_REPEAT_DIR, f'repeat_{int(repeat_idx):02d}_seed_{int(repeat_seed)}')


def _unsmear_ckpt_path(short_name: str, repeat_idx: int, repeat_seed: int) -> str:
    repeat_dir = _unsmear_repeat_dir(repeat_idx, repeat_seed)
    tool.ensure_dir(repeat_dir)
    return os.path.join(repeat_dir, f'unsmear_{short_name}.pt')


def _unsmear_history_path(short_name: str, repeat_idx: int, repeat_seed: int) -> str:
    repeat_dir = _unsmear_repeat_dir(repeat_idx, repeat_seed)
    tool.ensure_dir(repeat_dir)
    return os.path.join(repeat_dir, f'unsmear_{short_name}_history.csv')


def _make_epoch_train_loader_for_repeat(repeat_seed: int, epoch: int):
    if not bool(CONFIG['training'].get('resmear_each_epoch', False)):
        return train_loader
    stride = int(CONFIG['training'].get('resmear_seed_stride', 1))
    epoch_seed = int(repeat_seed + (int(epoch) - 1) * stride)
    x_ep, y_ep, m_ep = tool.build_unsmear_epoch_arrays(
        train_const_raw,
        train_mask_raw,
        hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=epoch_seed,
        clip=10.0,
    )
    ds = tool.UnsmearJetDataset(x_ep, y_ep, m_ep)
    return DataLoader(ds, batch_size=BS, shuffle=True, drop_last=True)


def _train_regression_repeat(name: str, model, *, repeat_seed: int):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = make_scheduler(opt)
    best_val, best_state, no_imp = 1e9, None, 0
    t0 = time.time()
    history_rows = []

    for ep in range(1, epochs + 1):
        model.train()
        tot = 0.0
        n = 0
        epoch_train_loader = _make_epoch_train_loader_for_repeat(repeat_seed, ep)
        for batch in epoch_train_loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)
            m = batch['mask'].to(device)

            opt.zero_grad(set_to_none=True)
            out = model(x, m)
            if isinstance(out, tuple):
                mu, log_var = out
            else:
                mu, log_var = out, None
            loss = regression_loss(mu, y, m, log_var=log_var)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tot += float(loss.item()) * int(x.shape[0])
            n += int(x.shape[0])

        train_loss = tot / max(1, n)
        val_loss = eval_regression(model, val_loader)
        sch.step()
        history_rows.append({'epoch': int(ep), 'train_loss': float(train_loss), 'val_loss': float(val_loss)})

        if val_loss < best_val - 1e-5:
            best_val = float(val_loss)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if ep == 1 or ep % 5 == 0:
            dt = time.time() - t0
            print(f'[{name}] ep={ep:03d} train={train_loss:.6f} val={val_loss:.6f} best={best_val:.6f} no_imp={no_imp} time={dt:.1f}s')
        if no_imp >= pat:
            print(f'[{name}] Early stopping')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss = eval_regression(model, test_loader)
    print(f'{name} test loss:', float(test_loss))
    return model, float(best_val), float(test_loss), history_rows


def _train_fm_repeat(model, *, repeat_seed: int):
    fm_steps = int(CONFIG['fm']['sample_steps'])
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = make_scheduler(opt)
    best_val, best_state, no_imp = 1e9, None, 0
    t0 = time.time()
    history_rows = []

    for ep in range(1, epochs + 1):
        model.train()
        tot = 0.0
        n = 0
        epoch_train_loader = _make_epoch_train_loader_for_repeat(repeat_seed, ep)
        for batch in epoch_train_loader:
            x_post = batch['x'].to(device)
            x_pre = batch['y'].to(device)
            m = batch['mask'].to(device)

            t = torch.rand((x_post.size(0),), device=device, dtype=x_post.dtype)
            x_t, v = tool.fm_make_bridge(
                x_post,
                x_pre,
                t,
                dphi_idx=dphi_idx,
                dphi_mean=dphi_mean,
                dphi_scale=dphi_scale,
            )

            opt.zero_grad(set_to_none=True)
            v_hat = model(x_t, x_post, m, t)
            loss = tool.masked_mse(v_hat, v, m)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tot += float(loss.item()) * int(x_post.shape[0])
            n += int(x_post.shape[0])

        train_loss = tot / max(1, n)
        val_loss = eval_fm(model, val_loader, steps=fm_steps)
        sch.step()
        history_rows.append({'epoch': int(ep), 'train_loss': float(train_loss), 'val_loss': float(val_loss)})

        if val_loss < best_val - 1e-5:
            best_val = float(val_loss)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if ep == 1 or ep % 5 == 0:
            dt = time.time() - t0
            print(f'[FM] ep={ep:03d} train={train_loss:.6f} val={val_loss:.6f} best={best_val:.6f} no_imp={no_imp} time={dt:.1f}s')
        if no_imp >= pat:
            print('[FM] Early stopping')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss = eval_fm(model, test_loader, steps=fm_steps)
    print('FM test loss:', float(test_loss))
    return model, float(best_val), float(test_loss), history_rows


def _load_unsmear_repeat_model(short_name: str, repeat_seed: int):
    repeat_idx = int(REPEAT_SEEDS.index(int(repeat_seed)))
    spec = next(item for item in UNSMEAR_MODEL_SPECS if item['short_name'] == str(short_name))
    model = _make_unsmear_model(spec)
    ckpt_path = _unsmear_ckpt_path(spec['short_name'], repeat_idx, repeat_seed)
    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f'Unsmear repeat checkpoint not found: {ckpt_path}')
    obj = torch.load(ckpt_path, map_location='cpu')
    state = obj['state_dict'] if isinstance(obj, dict) and 'state_dict' in obj else obj
    model.load_state_dict(state)
    print('Loaded repeat checkpoint:', ckpt_path)
    return model, ckpt_path


print()
print('=== Unsmear repeat training ===')
print('Repeat seeds:', REPEAT_SEEDS)

unsmear_repeat_rows = []

for repeat_idx, repeat_seed in enumerate(REPEAT_SEEDS):
    print()
    print(f'==== Unsmear repeat {repeat_idx + 1}/{len(REPEAT_SEEDS)} seed={repeat_seed} ====')
    _set_repeat_seed(int(repeat_seed))

    for spec in UNSMEAR_MODEL_SPECS:
        name = str(spec['display_name'])
        short_name = str(spec['short_name'])
        ckpt_path = _unsmear_ckpt_path(short_name, repeat_idx, repeat_seed)
        hist_path = _unsmear_history_path(short_name, repeat_idx, repeat_seed)
        model = _make_unsmear_model(spec)

        if bool(CONFIG.get('load_unsmear_bundle', False)) and os.path.isfile(ckpt_path):
            obj = torch.load(ckpt_path, map_location='cpu')
            state = obj['state_dict'] if isinstance(obj, dict) and 'state_dict' in obj else obj
            model.load_state_dict(state)
            best_val = float(obj.get('best_val', np.nan)) if isinstance(obj, dict) else float('nan')
            test_loss = float('nan')
            history_rows = []
            print('Loaded checkpoint:', ckpt_path)
        else:
            if spec['kind'] == 'fm':
                model, best_val, test_loss, history_rows = _train_fm_repeat(model, repeat_seed=int(repeat_seed))
            else:
                model, best_val, test_loss, history_rows = _train_regression_repeat(name, model, repeat_seed=int(repeat_seed))
            torch.save({'state_dict': model.state_dict(), 'best_val': best_val, 'test_loss': test_loss, 'config': CONFIG}, ckpt_path)
            print('Saved checkpoint:', ckpt_path)
            if history_rows:
                pd.DataFrame(history_rows).to_csv(hist_path, index=False)
                print('Saved history:', hist_path)

        unsmear_repeat_rows.append(
            {
                'repeat_idx': int(repeat_idx),
                'repeat_seed': int(repeat_seed),
                'model': name,
                'best_val_loss': float(best_val),
                'test_loss': float(test_loss),
                'ckpt_path': ckpt_path,
                'history_path': hist_path,
            }
        )

unsmear_repeat_df = pd.DataFrame(unsmear_repeat_rows)
unsmear_repeat_summary_rows = []
for model_name in ['MLP', 'Transformer', 'FM']:
    df_model = unsmear_repeat_df[unsmear_repeat_df['model'] == model_name].copy()
    if df_model.empty:
        continue
    unsmear_repeat_summary_rows.append(
        {
            'model': model_name,
            'num_repeats': int(len(df_model)),
            'best_val_loss_mean': float(df_model['best_val_loss'].mean()),
            'best_val_loss_std': float(df_model['best_val_loss'].std(ddof=0)),
            'test_loss_mean': float(df_model['test_loss'].mean()),
            'test_loss_std': float(df_model['test_loss'].std(ddof=0)),
        }
    )

unsmear_repeat_summary_df = pd.DataFrame(unsmear_repeat_summary_rows)
_save_rows_csv(unsmear_repeat_rows, os.path.join(UNSMEAR_REPEAT_TABLE_DIR, 'unsmear_repeat_detail.csv'))
_save_rows_csv(unsmear_repeat_summary_rows, os.path.join(UNSMEAR_REPEAT_TABLE_DIR, 'unsmear_repeat_summary.csv'))
print('Representative unsmear repeat seed:', REPRESENTATIVE_UNSMEAR_REPEAT_SEED)


In [ ]:
# Visualizations
# 用代表性 repeat 的上游 unsmear 模型做可视化，只保留 MLP / Transformer / FM。

@torch.no_grad()
def collect_all(loader):
    ys = []
    xs = []
    masks = []
    for batch in loader:
        x = batch['x'].to(device)
        m = batch['mask'].to(device)
        y = batch['y'].to(device)
        ys.append(y.cpu().numpy())
        xs.append(x.cpu().numpy())
        masks.append(m.cpu().numpy())

    return {
        'y': np.concatenate(ys, axis=0),
        'x': np.concatenate(xs, axis=0),
        'mask': np.concatenate(masks, axis=0),
    }


rep_mlp, rep_mlp_ckpt = _load_unsmear_repeat_model('mlp', REPRESENTATIVE_UNSMEAR_REPEAT_SEED)
rep_transformer, rep_transformer_ckpt = _load_unsmear_repeat_model('transformer', REPRESENTATIVE_UNSMEAR_REPEAT_SEED)
rep_fm, rep_fm_ckpt = _load_unsmear_repeat_model('fm', REPRESENTATIVE_UNSMEAR_REPEAT_SEED)

print('Representative MLP checkpoint:', rep_mlp_ckpt)
print('Representative Transformer checkpoint:', rep_transformer_ckpt)
print('Representative FM checkpoint:', rep_fm_ckpt)

pack = collect_all(test_loader)
y_std = pack['y']
x_std = pack['x']
mask_np = pack['mask']

pred_mlp = _predict_regression_arrays(rep_mlp, x_train[test_idx], post_mask[test_idx], bs=512)
pred_transformer = _predict_regression_arrays(rep_transformer, x_train[test_idx], post_mask[test_idx], bs=512)
pred_fm = _predict_fm_arrays(rep_fm, x_train[test_idx], post_mask[test_idx], bs=512)

# 1) per-feature residual distributions (before vs MLP vs Transformer vs FM)
for i, name in enumerate(feat_names):
    x_i = x_std[..., i][mask_np]
    y_i = y_std[..., i][mask_np]
    m_i = pred_mlp[..., i][mask_np]
    t_i = pred_transformer[..., i][mask_np]
    f_i = pred_fm[..., i][mask_np]

    r_before = x_i - y_i
    r_mlp = m_i - y_i
    r_transformer = t_i - y_i
    r_fm = f_i - y_i

    if name == 'dPhi':
        sc = float(feat_stds[i])
        r_before = tool.wrap_dphi_np(r_before * sc) / sc
        r_mlp = tool.wrap_dphi_np(r_mlp * sc) / sc
        r_transformer = tool.wrap_dphi_np(r_transformer * sc) / sc
        r_fm = tool.wrap_dphi_np(r_fm * sc) / sc

    plt.figure(figsize=(6.8, 4.6))
    bins = 120
    plt.hist(r_before, bins=bins, density=True, alpha=0.30, label='Before (post - pre)')
    plt.hist(r_mlp, bins=bins, density=True, alpha=0.30, label='MLP (pred - pre)')
    plt.hist(r_transformer, bins=bins, density=True, alpha=0.30, label='Transformer (pred - pre)')
    plt.hist(r_fm, bins=bins, density=True, alpha=0.30, label='FM (pred - pre)')
    plt.title(f'Residual compare: {name}')
    plt.xlabel('Residual (std space)')
    plt.ylabel('Density')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f'resid_compare_{name}.png')
    plt.savefig(fig_path, dpi=160, bbox_inches='tight')
    print('Saved figure:', fig_path)
    plt.show()

# 2) pred vs true scatter (MLP / Transformer / FM)
N_scatter = 100000
sel = np.where(mask_np.reshape(-1))[0]
if sel.size > N_scatter:
    sel = np.random.RandomState(0).choice(sel, size=N_scatter, replace=False)

for i, name in enumerate(feat_names):
    y_flat = y_std[..., i].reshape(-1)[sel]
    m_flat_pred = pred_mlp[..., i].reshape(-1)[sel]
    t_flat = pred_transformer[..., i].reshape(-1)[sel]
    f_flat = pred_fm[..., i].reshape(-1)[sel]

    lo = np.percentile(np.concatenate([y_flat, m_flat_pred, t_flat, f_flat]), 1)
    hi = np.percentile(np.concatenate([y_flat, m_flat_pred, t_flat, f_flat]), 99)

    plt.figure(figsize=(16.5, 4.8))
    plt.subplot(1, 3, 1)
    plt.hexbin(y_flat, m_flat_pred, gridsize=80, bins='log', mincnt=1)
    plt.plot([lo, hi], [lo, hi], 'r--', lw=1)
    plt.xlabel('True target (std)')
    plt.ylabel('MLP pred (std)')
    plt.title(f'MLP: {name}')
    plt.grid(True, alpha=0.2)

    plt.subplot(1, 3, 2)
    plt.hexbin(y_flat, t_flat, gridsize=80, bins='log', mincnt=1)
    plt.plot([lo, hi], [lo, hi], 'r--', lw=1)
    plt.xlabel('True target (std)')
    plt.ylabel('Transformer pred (std)')
    plt.title(f'Transformer: {name}')
    plt.grid(True, alpha=0.2)

    plt.subplot(1, 3, 3)
    plt.hexbin(y_flat, f_flat, gridsize=80, bins='log', mincnt=1)
    plt.plot([lo, hi], [lo, hi], 'r--', lw=1)
    plt.xlabel('True target (std)')
    plt.ylabel('FM pred (std)')
    plt.title(f'FM: {name}')
    plt.grid(True, alpha=0.2)

    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f'scatter_compare_{name}.png')
    plt.savefig(fig_path, dpi=160, bbox_inches='tight')
    print('Saved figure:', fig_path)
    plt.show()

# 3) Quantitative tables (MAE/RMSE + tail quantiles) and binned bias curves
res_before = x_std - y_std
res_mlp = pred_mlp - y_std
res_transformer = pred_transformer - y_std
res_fm = pred_fm - y_std

m_flat = mask_np.reshape(-1)
splits = [('all', m_flat)]


def _metrics(res_1d: np.ndarray):
    if res_1d.size == 0:
        return {'bias': np.nan, 'mae': np.nan, 'rmse': np.nan, 'p50': np.nan, 'p90': np.nan, 'p99': np.nan}
    abs_r = np.abs(res_1d)
    return {
        'bias': float(np.mean(res_1d)),
        'mae': float(np.mean(abs_r)),
        'rmse': float(np.sqrt(np.mean(res_1d ** 2))),
        'p50': float(np.quantile(abs_r, 0.50)),
        'p90': float(np.quantile(abs_r, 0.90)),
        'p99': float(np.quantile(abs_r, 0.99)),
    }


def print_metrics_table(split_name: str, sel_flat: np.ndarray):
    print('\n' + '=' * 80)
    print(f"Metrics summary (std space) | split={split_name} | n_tokens={int(sel_flat.sum())}")
    print('=' * 80)
    header = ['feature', 'method', 'bias', 'mae', 'rmse', 'abs_p50', 'abs_p90', 'abs_p99']
    print('\t'.join(header))

    for i, name in enumerate(feat_names):
        y_flat = y_std[..., i].reshape(-1)[sel_flat]
        if y_flat.size == 0:
            continue
        for method, res in [('before', res_before), ('mlp', res_mlp), ('transformer', res_transformer), ('fm', res_fm)]:
            r = res[..., i].reshape(-1)[sel_flat]
            if name == 'dPhi':
                sc = float(feat_stds[i])
                r = tool.wrap_dphi_np(r * sc) / sc
            mm = _metrics(r)
            print('\t'.join([
                name,
                method,
                f"{mm['bias']:.6f}",
                f"{mm['mae']:.6f}",
                f"{mm['rmse']:.6f}",
                f"{mm['p50']:.6f}",
                f"{mm['p90']:.6f}",
                f"{mm['p99']:.6f}",
            ]))


for split_name, sel_flat in splits:
    print_metrics_table(split_name, sel_flat)


def plot_binned_bias(feature_idx: int, feature_name: str, *, sel_flat: np.ndarray, split_name: str, n_bins: int = 20):
    y_flat = y_std[..., feature_idx].reshape(-1)[sel_flat]
    if y_flat.size < 1000:
        return

    qs = np.linspace(0.0, 1.0, int(n_bins) + 1)
    edges = np.quantile(y_flat, qs)
    edges = np.unique(edges)
    if edges.size < 3:
        return

    centers = 0.5 * (edges[:-1] + edges[1:])

    def _binned_stats(res_arr: np.ndarray):
        r_flat = res_arr[..., feature_idx].reshape(-1)[sel_flat]
        if feature_name == 'dPhi':
            sc = float(feat_stds[int(feature_idx)])
            r_flat = tool.wrap_dphi_np(r_flat * sc) / sc
        means = []
        stds = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            inb = (y_flat >= lo) & (y_flat < hi)
            if not np.any(inb):
                means.append(np.nan)
                stds.append(np.nan)
            else:
                means.append(float(np.mean(r_flat[inb])))
                stds.append(float(np.std(r_flat[inb])))
        return np.asarray(means), np.asarray(stds)

    mb, sb = _binned_stats(res_before)
    mm, sm = _binned_stats(res_mlp)
    mt, st = _binned_stats(res_transformer)
    mf, sf = _binned_stats(res_fm)

    plt.figure(figsize=(7.2, 4.8))
    plt.plot(centers, mb, label='Before', lw=2)
    plt.fill_between(centers, mb - sb, mb + sb, alpha=0.12)

    plt.plot(centers, mm, label='MLP', lw=2)
    plt.fill_between(centers, mm - sm, mm + sm, alpha=0.12)

    plt.plot(centers, mt, label='Transformer', lw=2)
    plt.fill_between(centers, mt - st, mt + st, alpha=0.12)

    plt.plot(centers, mf, label='FM', lw=2)
    plt.fill_between(centers, mf - sf, mf + sf, alpha=0.12)

    plt.axhline(0.0, color='k', lw=1, alpha=0.6)
    plt.xlabel(f'True {feature_name} (std)')
    plt.ylabel('Mean residual (pred - true)')
    plt.title(f'Binned bias: {feature_name} | {split_name}')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()

    out = os.path.join(FIG_DIR, f'bias_{feature_name}_{split_name}.png')
    plt.savefig(out, dpi=160, bbox_inches='tight')
    print('Saved figure:', out)
    plt.show()


for split_name, sel_flat in splits:
    for i, name in enumerate(feat_names):
        plot_binned_bias(i, name, sel_flat=sel_flat, split_name=split_name, n_bins=20)


In [ ]:
# Downstream tagger evaluation
# 把整条方法链做成真正的 repeat：上游 unsmear 模型与下游 tagger 都按相同 seed 跑三次。
# offline / hlt / hlt_kd 直接从外部 Baseline_smearing repeat 结果读取。

from model import ParticleTransformerKD  # noqa: E402

baseline_tool = tool

# 显式接出下游 tagger / KD 配置，避免依赖旧 kernel 里的残留变量。
TAGGER_CFG = dict(CONFIG['tagger'])
TRAIN_TAG = dict(CONFIG['tagger_training'])
KD_CFG = {'kd': dict(CONFIG['tagger_kd'])}
BS_TAG = int(TRAIN_TAG['batch_size'])
DOWN_KIND = str(CONFIG.get('downstream_feature_kind', '7d')).lower()

KD_TEACHER_REPEAT_SEED = int(CONFIG.get('kd_teacher_repeat_seed', 62))
BASELINE_REPEAT_ROOT = str(CONFIG.get('external_baseline_repeat_dir', EXTERNAL_BASELINE_REPEAT_DIR))
DOWNSTREAM_REPEAT_DIR = os.path.join(CKPT_DIR, 'downstream_tagger_repeats')
DOWNSTREAM_REPEAT_FIG_DIR = os.path.join(DOWNSTREAM_REPEAT_DIR, 'figs')
DOWNSTREAM_REPEAT_TABLE_DIR = os.path.join(DOWNSTREAM_REPEAT_DIR, 'tables')
tool.ensure_dir(DOWNSTREAM_REPEAT_DIR)
tool.ensure_dir(DOWNSTREAM_REPEAT_FIG_DIR)
tool.ensure_dir(DOWNSTREAM_REPEAT_TABLE_DIR)

print('Downstream tagger cfg:', TAGGER_CFG)
print('Downstream train cfg:', TRAIN_TAG)
print('Downstream KD cfg:', KD_CFG['kd'])


def _make_loader(ds, *, shuffle: bool, drop_last: bool = False, seed_value: int | None = None):
    gen = None
    if seed_value is not None:
        gen = torch.Generator()
        gen.manual_seed(int(seed_value))
    return DataLoader(
        ds,
        batch_size=int(BS_TAG),
        shuffle=bool(shuffle),
        drop_last=bool(drop_last),
        generator=gen,
    )


def _load_state_only(model, ckpt_path: str):
    obj = torch.load(ckpt_path, map_location='cpu')
    state = obj['state_dict'] if isinstance(obj, dict) and 'state_dict' in obj else obj
    model.load_state_dict(state)
    print('Loaded checkpoint:', ckpt_path)
    return obj if isinstance(obj, dict) else {'state_dict': state}


def _save_state_only(model, ckpt_path: str, *, extra: dict | None = None):
    payload = {
        'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
    }
    if extra is not None:
        payload['extra'] = dict(extra)
    torch.save(payload, ckpt_path)
    print('Saved checkpoint:', ckpt_path)


def _save_history(history_rows: list[dict], out_path: str):
    pd.DataFrame(history_rows).to_csv(out_path, index=False)
    print('Saved history:', out_path)


@torch.no_grad()
def _evaluate_model_full(model, loader, *, feat_key: str, mask_key: str):
    model.eval()
    preds, labels_eval, weights_eval = [], [], []
    for batch in loader:
        x = batch[feat_key].to(device)
        m = batch[mask_key].to(device)
        y = batch['label'].to(device)
        w = batch['weight'].to(device)
        logits = model(x, m).squeeze(-1)
        preds.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())
        labels_eval.extend(y.detach().cpu().numpy().tolist())
        weights_eval.extend(w.detach().cpu().numpy().tolist())
    preds = np.asarray(preds, dtype=np.float32)
    labels_eval = np.asarray(labels_eval, dtype=np.float32)
    weights_eval = np.asarray(weights_eval, dtype=np.float32)
    auc = float(baseline_tool.roc_auc_score(labels_eval, preds))
    return auc, preds, labels_eval, weights_eval


def _save_prediction_bundle(pred_path: str, *, preds: np.ndarray, labels_eval: np.ndarray, weights_eval: np.ndarray):
    np.savez_compressed(
        pred_path,
        preds=np.asarray(preds, dtype=np.float32),
        labels=np.asarray(labels_eval, dtype=np.float32),
        weights=np.asarray(weights_eval, dtype=np.float32),
    )
    print('Saved prediction bundle:', pred_path)


def _find_external_repeat_dir(repeat_seed: int) -> str:
    if not os.path.isdir(BASELINE_REPEAT_ROOT):
        raise FileNotFoundError(f'External baseline repeat dir not found: {BASELINE_REPEAT_ROOT}')
    suffix = f'seed_{int(repeat_seed)}'
    matches = []
    for name in sorted(os.listdir(BASELINE_REPEAT_ROOT)):
        full = os.path.join(BASELINE_REPEAT_ROOT, name)
        if os.path.isdir(full) and name.endswith(suffix):
            matches.append(full)
    if not matches:
        raise FileNotFoundError(f'Cannot find external repeat dir for seed={repeat_seed} under {BASELINE_REPEAT_ROOT}')
    if len(matches) > 1:
        raise RuntimeError(f'Multiple external repeat dirs matched seed={repeat_seed}: {matches}')
    return matches[0]


EXTERNAL_BASELINE_FILES = {
    'Teacher(OFF_FULL)': {
        'ckpt': 'offline_teacher.pt',
        'pred': 'offline_teacher_test_preds.npz',
    },
    'Student(HLT)': {
        'ckpt': 'hlt_baseline.pt',
        'pred': 'hlt_baseline_test_preds.npz',
    },
    'Student(HLT)+KD': {
        'ckpt': 'hlt_kd_student.pt',
        'pred': 'hlt_kd_student_test_preds.npz',
    },
}


def _load_external_baseline_result(repeat_seed: int, model_name: str) -> dict:
    repeat_dir = _find_external_repeat_dir(int(repeat_seed))
    spec = EXTERNAL_BASELINE_FILES[str(model_name)]
    ckpt_path = os.path.join(repeat_dir, 'ckpts', spec['ckpt'])
    pred_path = os.path.join(repeat_dir, 'predictions', spec['pred'])
    if not os.path.isfile(pred_path):
        raise FileNotFoundError(f'External prediction bundle missing: {pred_path}')
    bundle = np.load(pred_path)
    preds = np.asarray(bundle['preds'], dtype=np.float32)
    labels_eval = np.asarray(bundle['labels'], dtype=np.float32)
    weights_eval = np.asarray(bundle['weights'], dtype=np.float32) if 'weights' in bundle.files else np.ones_like(labels_eval, dtype=np.float32)
    auc = float(baseline_tool.roc_auc_score(labels_eval, preds))
    print(f'Loaded external baseline result: seed={repeat_seed} model={model_name}')
    return {
        'model': str(model_name),
        'source': 'external_baseline',
        'repeat_seed': int(repeat_seed),
        'auc': float(auc),
        'preds': preds,
        'labels': labels_eval,
        'weights': weights_eval,
        'ckpt_path': ckpt_path,
        'prediction_path': pred_path,
    }


def _load_kd_teacher_from_external_repeat(repeat_seed: int):
    teacher_model = ParticleTransformerKD(**TAGGER_CFG).to(device)
    repeat_dir = _find_external_repeat_dir(int(repeat_seed))
    ckpt_path = os.path.join(repeat_dir, 'ckpts', EXTERNAL_BASELINE_FILES['Teacher(OFF_FULL)']['ckpt'])
    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f'KD teacher checkpoint missing: {ckpt_path}')
    _load_state_only(teacher_model, ckpt_path)
    teacher_model.eval()
    return teacher_model, ckpt_path


def _fpr_at_target_tpr(y_true: np.ndarray, y_score: np.ndarray, target_tpr: float) -> float:
    fpr, tpr, _ = baseline_tool.compute_roc(y_true, y_score)
    tpr = np.asarray(tpr, dtype=np.float64)
    fpr = np.asarray(fpr, dtype=np.float64)
    unique_tpr, unique_idx = np.unique(tpr, return_index=True)
    unique_fpr = fpr[unique_idx]
    return float(np.interp(float(target_tpr), unique_tpr, unique_fpr))


def _gap_recovery(model_fpr: float, baseline_fpr: float, teacher_fpr: float) -> float:
    denom = float(baseline_fpr - teacher_fpr)
    if abs(denom) < 1e-12:
        return float('nan')
    return float((baseline_fpr - model_fpr) / denom)


def _build_repeat_unsmear_std_features(repeat_seed: int):
    repeat_seed = int(repeat_seed)
    models = {}
    for short_name in ['mlp', 'transformer', 'fm']:
        model, _ = _load_unsmear_repeat_model(short_name, repeat_seed)
        models[short_name] = model

    preds_std = {
        'mlp': {
            'train': _predict_regression_arrays(models['mlp'], x_train[train_idx], post_mask[train_idx], bs=512),
            'val': _predict_regression_arrays(models['mlp'], x_train[val_idx], post_mask[val_idx], bs=512),
            'test': _predict_regression_arrays(models['mlp'], x_train[test_idx], post_mask[test_idx], bs=512),
        },
        'transformer': {
            'train': _predict_regression_arrays(models['transformer'], x_train[train_idx], post_mask[train_idx], bs=512),
            'val': _predict_regression_arrays(models['transformer'], x_train[val_idx], post_mask[val_idx], bs=512),
            'test': _predict_regression_arrays(models['transformer'], x_train[test_idx], post_mask[test_idx], bs=512),
        },
        'fm': {
            'train': _predict_fm_arrays(models['fm'], x_train[train_idx], post_mask[train_idx], bs=512),
            'val': _predict_fm_arrays(models['fm'], x_train[val_idx], post_mask[val_idx], bs=512),
            'test': _predict_fm_arrays(models['fm'], x_train[test_idx], post_mask[test_idx], bs=512),
        },
    }

    def _slice_axis(axis: dict, idx: np.ndarray) -> dict:
        return {k: np.asarray(v)[idx] for k, v in axis.items()}

    def _restore_pred_raw(pred_std: np.ndarray) -> np.ndarray:
        raw = pred_std * feat_stds[None, None, :] + feat_means[None, None, :]
        raw = np.asarray(raw, dtype=np.float32)
        if raw.shape[-1] > 1:
            raw[..., 1] = tool.wrap_dphi_np(raw[..., 1])
        return raw

    def _rebuild_post3_from_first4(raw_pred: np.ndarray, axis_split: dict, mask_split: np.ndarray) -> np.ndarray:
        rebuilt = raw_pred.copy().astype(np.float32)
        rebuilt7 = tool.feats_to_7d(raw_pred[..., :4], mask_split, axis_split, kind='4d')
        rebuilt[..., 4:7] = rebuilt7[..., 4:7]
        rebuilt[~mask_split] = 0.0
        return rebuilt.astype(np.float32)

    axis_off_train = _slice_axis(axis_off, train_idx)
    axis_off_val = _slice_axis(axis_off, val_idx)
    axis_off_test = _slice_axis(axis_off, test_idx)

    outputs = {}
    for short_name, display_name in [('mlp', 'Student(MLP+HLT)'), ('transformer', 'Student(Transformer+HLT)'), ('fm', 'Student(FM+HLT)')]:
        raw_direct = {
            split: _restore_pred_raw(preds_std[short_name][split])
            for split in ['train', 'val', 'test']
        }
        if DOWN_KIND == '7d' and str(CONFIG['feature_kind']).lower() == '7d':
            if POST3_SOURCE == 'rebuild_from_first4':
                raw_split = {
                    'train': _rebuild_post3_from_first4(raw_direct['train'], axis_off_train, post_mask[train_idx]),
                    'val': _rebuild_post3_from_first4(raw_direct['val'], axis_off_val, post_mask[val_idx]),
                    'test': _rebuild_post3_from_first4(raw_direct['test'], axis_off_test, post_mask[test_idx]),
                }
            else:
                raw_split = raw_direct
        elif DOWN_KIND == '7d' and str(CONFIG['feature_kind']).lower() != '7d':
            uns_kind = str(CONFIG['feature_kind']).lower()
            raw_split = {
                'train': tool.feats_to_7d(raw_direct['train'], post_mask[train_idx], axis_off_train, kind=uns_kind),
                'val': tool.feats_to_7d(raw_direct['val'], post_mask[val_idx], axis_off_val, kind=uns_kind),
                'test': tool.feats_to_7d(raw_direct['test'], post_mask[test_idx], axis_off_test, kind=uns_kind),
            }
        else:
            raw_split = raw_direct

        outputs[display_name] = {
            'train': tool.standardize(raw_split['train'], post_mask[train_idx], off_means, off_stds, clip=10.0),
            'val': tool.standardize(raw_split['val'], post_mask[val_idx], off_means, off_stds, clip=10.0),
            'test': tool.standardize(raw_split['test'], post_mask[test_idx], off_means, off_stds, clip=10.0),
        }
    return outputs


def _make_opt(model):
    opt = torch.optim.AdamW(model.parameters(), lr=float(TRAIN_TAG['lr']), weight_decay=float(TRAIN_TAG['weight_decay']))
    sch = baseline_tool.get_scheduler(opt, int(TRAIN_TAG['warmup_epochs']), int(TRAIN_TAG['epochs']))
    return opt, sch


def _train_standard(name: str, model, train_loader, val_loader, *, feat_key: str, mask_key: str):
    opt, sch = _make_opt(model)
    best_auc, best_state, no_imp = 0.0, None, 0
    history_rows = []
    for ep in range(1, int(TRAIN_TAG['epochs']) + 1):
        loss, train_auc = baseline_tool.train_standard(model, train_loader, opt, device, feat_key, mask_key)
        sch.step()
        val_auc, _, _ = baseline_tool.evaluate(model, val_loader, device, feat_key, mask_key)
        history_rows.append({'epoch': int(ep), 'train_loss': float(loss), 'train_auc': float(train_auc), 'val_auc': float(val_auc)})
        if val_auc > best_auc + 1e-4:
            best_auc = float(val_auc)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if ep % 2 == 0 or ep == 1:
            print(f'[{name}] ep={ep:03d} train_loss={loss:.5f} train_auc={train_auc:.5f} val_auc={val_auc:.5f} best={best_auc:.5f} no_imp={no_imp}')
        if no_imp >= int(TRAIN_TAG['patience']):
            print(f'[{name}] Early stopping')
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, float(best_auc), history_rows


def _train_kd(name: str, student, teacher, train_loader, val_loader):
    opt, sch = _make_opt(student)
    best_auc, best_state, no_imp = 0.0, None, 0
    history_rows = []
    for ep in range(1, int(TRAIN_TAG['epochs']) + 1):
        loss, train_auc = baseline_tool.train_kd(student, teacher, train_loader, opt, device, KD_CFG)
        sch.step()
        val_auc, _, _ = baseline_tool.evaluate(student, val_loader, device, 'hlt', 'mask_hlt')
        history_rows.append({'epoch': int(ep), 'train_loss': float(loss), 'train_auc': float(train_auc), 'val_auc': float(val_auc)})
        if val_auc > best_auc + 1e-4:
            best_auc = float(val_auc)
            best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if ep % 2 == 0 or ep == 1:
            print(f'[{name}] ep={ep:03d} train_loss={loss:.5f} train_auc={train_auc:.5f} val_auc={val_auc:.5f} best={best_auc:.5f} no_imp={no_imp}')
        if no_imp >= int(TRAIN_TAG['patience']):
            print(f'[{name}] Early stopping')
            break
    if best_state is not None:
        student.load_state_dict(best_state)
    return student, float(best_auc), history_rows


MODEL_DISPLAY_ORDER = [
    'Teacher(OFF_FULL)',
    'Student(HLT)',
    'Student(HLT)+KD',
    'Student(MLP+HLT)',
    'Student(Transformer+HLT)',
    'Student(FM+HLT)',
    'Student(MLP+HLT)+KD',
    'Student(Transformer+HLT)+KD',
    'Student(FM+HLT)+KD',
]

MODEL_COLORS = {
    'Teacher(OFF_FULL)': '#4C78A8',
    'Student(HLT)': '#F58518',
    'Student(HLT)+KD': '#54A24B',
    'Student(MLP+HLT)': '#E45756',
    'Student(Transformer+HLT)': '#72B7B2',
    'Student(FM+HLT)': '#FF9DA6',
    'Student(MLP+HLT)+KD': '#9D755D',
    'Student(Transformer+HLT)+KD': '#BAB0AC',
    'Student(FM+HLT)+KD': '#BC5090',
}


def _repeat_ckpt_name(short_name: str, *, kd: bool) -> str:
    suffix = '_kd' if bool(kd) else ''
    return f'tagger_{short_name}{suffix}_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}.pt'


def _repeat_pred_name(short_name: str, *, kd: bool) -> str:
    suffix = '_kd' if bool(kd) else ''
    return f'tagger_{short_name}{suffix}_{DOWN_KIND}_uns{str(CONFIG.get("feature_kind")).lower()}_test_preds.npz'


teacher_for_kd, teacher_for_kd_ckpt = _load_kd_teacher_from_external_repeat(KD_TEACHER_REPEAT_SEED)
print('KD teacher checkpoint:', teacher_for_kd_ckpt)

repeat_results = []

for repeat_idx, repeat_seed in enumerate(REPEAT_SEEDS):
    print()
    print(f'==== Full-chain downstream repeat {repeat_idx + 1}/{len(REPEAT_SEEDS)} seed={repeat_seed} ====')
    _set_repeat_seed(int(repeat_seed))

    repeat_dir = os.path.join(DOWNSTREAM_REPEAT_DIR, f'repeat_{int(repeat_idx):02d}_seed_{int(repeat_seed)}')
    repeat_ckpt_dir = os.path.join(repeat_dir, 'ckpts')
    repeat_pred_dir = os.path.join(repeat_dir, 'predictions')
    repeat_hist_dir = os.path.join(repeat_dir, 'history')
    tool.ensure_dir(repeat_ckpt_dir)
    tool.ensure_dir(repeat_pred_dir)
    tool.ensure_dir(repeat_hist_dir)

    repeat_model_results = {}
    for external_name in ['Teacher(OFF_FULL)', 'Student(HLT)', 'Student(HLT)+KD']:
        repeat_model_results[external_name] = _load_external_baseline_result(int(repeat_seed), external_name)

    repeat_unsmear_std = _build_repeat_unsmear_std_features(int(repeat_seed))

    local_method_specs = [
        {
            'display_name': 'Student(MLP+HLT)',
            'short_name': 'mlp',
            'train_ds': baseline_tool.JetDataset(feat_off_full_std[train_idx], repeat_unsmear_std['Student(MLP+HLT)']['train'], y_tr, m_off_train, m_hlt_train, w_tr),
            'val_ds': baseline_tool.JetDataset(feat_off_full_std[val_idx], repeat_unsmear_std['Student(MLP+HLT)']['val'], y_va, m_off_val, m_hlt_val, w_va),
            'test_ds': baseline_tool.JetDataset(feat_off_full_std[test_idx], repeat_unsmear_std['Student(MLP+HLT)']['test'], y_te, m_off_test, m_hlt_test, w_te),
        },
        {
            'display_name': 'Student(Transformer+HLT)',
            'short_name': 'transformer',
            'train_ds': baseline_tool.JetDataset(feat_off_full_std[train_idx], repeat_unsmear_std['Student(Transformer+HLT)']['train'], y_tr, m_off_train, m_hlt_train, w_tr),
            'val_ds': baseline_tool.JetDataset(feat_off_full_std[val_idx], repeat_unsmear_std['Student(Transformer+HLT)']['val'], y_va, m_off_val, m_hlt_val, w_va),
            'test_ds': baseline_tool.JetDataset(feat_off_full_std[test_idx], repeat_unsmear_std['Student(Transformer+HLT)']['test'], y_te, m_off_test, m_hlt_test, w_te),
        },
        {
            'display_name': 'Student(FM+HLT)',
            'short_name': 'fm',
            'train_ds': baseline_tool.JetDataset(feat_off_full_std[train_idx], repeat_unsmear_std['Student(FM+HLT)']['train'], y_tr, m_off_train, m_hlt_train, w_tr),
            'val_ds': baseline_tool.JetDataset(feat_off_full_std[val_idx], repeat_unsmear_std['Student(FM+HLT)']['val'], y_va, m_off_val, m_hlt_val, w_va),
            'test_ds': baseline_tool.JetDataset(feat_off_full_std[test_idx], repeat_unsmear_std['Student(FM+HLT)']['test'], y_te, m_off_test, m_hlt_test, w_te),
        },
    ]

    for spec in local_method_specs:
        train_loader = _make_loader(spec['train_ds'], shuffle=True, drop_last=True, seed_value=int(repeat_seed))
        val_loader = _make_loader(spec['val_ds'], shuffle=False)
        test_loader = _make_loader(spec['test_ds'], shuffle=False)

        standard_model = ParticleTransformerKD(**TAGGER_CFG).to(device)
        standard_ckpt = os.path.join(repeat_ckpt_dir, _repeat_ckpt_name(spec['short_name'], kd=False))
        standard_hist = os.path.join(repeat_hist_dir, _repeat_ckpt_name(spec['short_name'], kd=False).replace('.pt', '_history.csv'))
        if bool(CONFIG.get('load_unsmear_bundle', False)) and os.path.isfile(standard_ckpt):
            _load_state_only(standard_model, standard_ckpt)
            standard_best_val_auc = float('nan')
        else:
            standard_model, standard_best_val_auc, standard_history_rows = _train_standard(
                spec['display_name'],
                standard_model,
                train_loader,
                val_loader,
                feat_key='hlt',
                mask_key='mask_hlt',
            )
            _save_state_only(standard_model, standard_ckpt, extra={'repeat_seed': int(repeat_seed), 'best_val_auc': float(standard_best_val_auc), 'model_name': spec['display_name']})
            _save_history(standard_history_rows, standard_hist)
        auc_standard, preds_standard, labels_standard, weights_standard = _evaluate_model_full(standard_model, test_loader, feat_key='hlt', mask_key='mask_hlt')
        pred_standard = os.path.join(repeat_pred_dir, _repeat_pred_name(spec['short_name'], kd=False))
        _save_prediction_bundle(pred_standard, preds=preds_standard, labels_eval=labels_standard, weights_eval=weights_standard)
        repeat_model_results[spec['display_name']] = {
            'model': spec['display_name'],
            'source': 'local_unsmear',
            'repeat_seed': int(repeat_seed),
            'auc': float(auc_standard),
            'preds': preds_standard,
            'labels': labels_standard,
            'weights': weights_standard,
            'best_val_auc': float(standard_best_val_auc),
            'ckpt_path': standard_ckpt,
            'prediction_path': pred_standard,
            'history_path': standard_hist,
        }

        kd_display_name = spec['display_name'] + '+KD'
        kd_model = ParticleTransformerKD(**TAGGER_CFG).to(device)
        kd_ckpt = os.path.join(repeat_ckpt_dir, _repeat_ckpt_name(spec['short_name'], kd=True))
        kd_hist = os.path.join(repeat_hist_dir, _repeat_ckpt_name(spec['short_name'], kd=True).replace('.pt', '_history.csv'))
        if bool(CONFIG.get('load_unsmear_bundle', False)) and os.path.isfile(kd_ckpt):
            _load_state_only(kd_model, kd_ckpt)
            kd_best_val_auc = float('nan')
        else:
            kd_model, kd_best_val_auc, kd_history_rows = _train_kd(kd_display_name, kd_model, teacher_for_kd, train_loader, val_loader)
            _save_state_only(kd_model, kd_ckpt, extra={'repeat_seed': int(repeat_seed), 'best_val_auc': float(kd_best_val_auc), 'teacher_repeat_seed': int(KD_TEACHER_REPEAT_SEED), 'teacher_ckpt_path': str(teacher_for_kd_ckpt), 'model_name': kd_display_name})
            _save_history(kd_history_rows, kd_hist)
        auc_kd, preds_kd, labels_kd, weights_kd = _evaluate_model_full(kd_model, test_loader, feat_key='hlt', mask_key='mask_hlt')
        pred_kd = os.path.join(repeat_pred_dir, _repeat_pred_name(spec['short_name'], kd=True))
        _save_prediction_bundle(pred_kd, preds=preds_kd, labels_eval=labels_kd, weights_eval=weights_kd)
        repeat_model_results[kd_display_name] = {
            'model': kd_display_name,
            'source': 'local_unsmear_kd',
            'repeat_seed': int(repeat_seed),
            'auc': float(auc_kd),
            'preds': preds_kd,
            'labels': labels_kd,
            'weights': weights_kd,
            'best_val_auc': float(kd_best_val_auc),
            'teacher_repeat_seed': int(KD_TEACHER_REPEAT_SEED),
            'teacher_ckpt_path': str(teacher_for_kd_ckpt),
            'ckpt_path': kd_ckpt,
            'prediction_path': pred_kd,
            'history_path': kd_hist,
        }

    repeat_results.append({'repeat_idx': int(repeat_idx), 'repeat_seed': int(repeat_seed), 'models': repeat_model_results})

print()
print('Completed full-chain downstream repeats:', len(repeat_results))

downstream_repeat_detail_rows = []
for repeat_result in repeat_results:
    repeat_seed = int(repeat_result['repeat_seed'])
    teacher_result = repeat_result['models']['Teacher(OFF_FULL)']
    baseline_result = repeat_result['models']['Student(HLT)']
    teacher_fpr_at_30 = _fpr_at_target_tpr(teacher_result['labels'], teacher_result['preds'], 0.30)
    teacher_fpr_at_50 = _fpr_at_target_tpr(teacher_result['labels'], teacher_result['preds'], 0.50)
    baseline_fpr_at_30 = _fpr_at_target_tpr(baseline_result['labels'], baseline_result['preds'], 0.30)
    baseline_fpr_at_50 = _fpr_at_target_tpr(baseline_result['labels'], baseline_result['preds'], 0.50)

    for model_name in MODEL_DISPLAY_ORDER:
        if model_name not in repeat_result['models']:
            continue
        model_result = repeat_result['models'][model_name]
        fpr_at_30 = _fpr_at_target_tpr(model_result['labels'], model_result['preds'], 0.30)
        fpr_at_50 = _fpr_at_target_tpr(model_result['labels'], model_result['preds'], 0.50)
        downstream_repeat_detail_rows.append(
            {
                'repeat_seed': int(repeat_seed),
                'model': str(model_name),
                'source': str(model_result.get('source', 'unknown')),
                'auc': float(model_result['auc']),
                'fpr_at_tpr_30': float(fpr_at_30),
                'fpr_at_tpr_50': float(fpr_at_50),
                'gap_recovery_tpr_30': _gap_recovery(float(fpr_at_30), float(baseline_fpr_at_30), float(teacher_fpr_at_30)),
                'gap_recovery_tpr_50': _gap_recovery(float(fpr_at_50), float(baseline_fpr_at_50), float(teacher_fpr_at_50)),
                'prediction_path': str(model_result.get('prediction_path', '')),
                'ckpt_path': str(model_result.get('ckpt_path', '')),
                'teacher_repeat_seed': model_result.get('teacher_repeat_seed', np.nan),
                'teacher_ckpt_path': str(model_result.get('teacher_ckpt_path', '')),
            }
        )

downstream_repeat_detail_df = pd.DataFrame(downstream_repeat_detail_rows)
downstream_repeat_summary_rows = []
for model_name in MODEL_DISPLAY_ORDER:
    df_model = downstream_repeat_detail_df[downstream_repeat_detail_df['model'] == str(model_name)].copy()
    if df_model.empty:
        continue
    downstream_repeat_summary_rows.append(
        {
            'model': str(model_name),
            'num_repeats': int(len(df_model)),
            'auc_mean': float(df_model['auc'].mean()),
            'auc_std': float(df_model['auc'].std(ddof=0)),
            'fpr_at_tpr_30_mean': float(df_model['fpr_at_tpr_30'].mean()),
            'fpr_at_tpr_30_std': float(df_model['fpr_at_tpr_30'].std(ddof=0)),
            'fpr_at_tpr_50_mean': float(df_model['fpr_at_tpr_50'].mean()),
            'fpr_at_tpr_50_std': float(df_model['fpr_at_tpr_50'].std(ddof=0)),
            'gap_recovery_tpr_30_mean': float(df_model['gap_recovery_tpr_30'].mean()),
            'gap_recovery_tpr_30_std': float(df_model['gap_recovery_tpr_30'].std(ddof=0)),
            'gap_recovery_tpr_50_mean': float(df_model['gap_recovery_tpr_50'].mean()),
            'gap_recovery_tpr_50_std': float(df_model['gap_recovery_tpr_50'].std(ddof=0)),
        }
    )

downstream_repeat_summary_df = pd.DataFrame(downstream_repeat_summary_rows)

plot_tpr_grid = np.linspace(0.0, 1.0, 500)
plt.figure(figsize=(9.2, 6.6))
for model_name in MODEL_DISPLAY_ORDER:
    curve_rows = []
    auc_values = []
    for repeat_result in repeat_results:
        if model_name not in repeat_result['models']:
            continue
        model_result = repeat_result['models'][model_name]
        fpr_curve, tpr_curve, _ = baseline_tool.compute_roc(model_result['labels'], model_result['preds'])
        tpr_curve = np.asarray(tpr_curve, dtype=np.float64)
        fpr_curve = np.asarray(fpr_curve, dtype=np.float64)
        unique_tpr, unique_idx = np.unique(tpr_curve, return_index=True)
        unique_fpr = fpr_curve[unique_idx]
        interp_fpr = np.interp(plot_tpr_grid, unique_tpr, unique_fpr)
        curve_rows.append(interp_fpr)
        auc_values.append(float(model_result['auc']))
    if not curve_rows:
        continue
    curve_arr = np.asarray(curve_rows, dtype=np.float64)
    mean_fpr = curve_arr.mean(axis=0)
    std_fpr = curve_arr.std(axis=0, ddof=0)
    mean_auc = float(np.mean(np.asarray(auc_values, dtype=np.float64)))
    std_auc = float(np.std(np.asarray(auc_values, dtype=np.float64), ddof=0))
    color = MODEL_COLORS.get(model_name, None)
    plt.semilogy(plot_tpr_grid, np.clip(mean_fpr, 1e-6, 1.0), lw=2, color=color, label=f'{model_name} (AUC={mean_auc:.4f}±{std_auc:.4f})')
    plt.fill_between(plot_tpr_grid, np.clip(mean_fpr - std_fpr, 1e-6, 1.0), np.clip(mean_fpr + std_fpr, 1e-6, 1.0), color=color, alpha=0.10)

plt.xlabel('TPR')
plt.ylabel('FPR (log)')
plt.title('Downstream ROC mean/std across repeats')
plt.grid(True, which='both', alpha=0.25)
plt.legend(loc='upper right', fontsize=8)
plt.xlim(0.0, 1.0)
plt.ylim(1e-4, 1.0)

downstream_roc_out = os.path.join(DOWNSTREAM_REPEAT_FIG_DIR, 'downstream_roc_logfpr_mean_std.png')
plt.tight_layout()
plt.savefig(downstream_roc_out, dpi=170, bbox_inches='tight')
print('Saved figure:', downstream_roc_out)
plt.show()


In [ ]:
# 保存并展示 repeat 版的下游指标表。
detail_out = os.path.join(DOWNSTREAM_REPEAT_TABLE_DIR, 'downstream_repeat_metrics_detail.csv')
summary_out = os.path.join(DOWNSTREAM_REPEAT_TABLE_DIR, 'downstream_repeat_metrics_summary.csv')

downstream_repeat_detail_df.to_csv(detail_out, index=False)
downstream_repeat_summary_df.to_csv(summary_out, index=False)

detail_display = downstream_repeat_detail_df.copy()
summary_display = downstream_repeat_summary_df.copy()

if not detail_display.empty:
    for col in ['auc', 'fpr_at_tpr_30', 'fpr_at_tpr_50', 'gap_recovery_tpr_30', 'gap_recovery_tpr_50']:
        detail_display[col] = detail_display[col].map(lambda x: 'nan' if pd.isna(x) else f'{float(x):.6e}' if 'fpr' in col else f'{float(x):.6f}')

if not summary_display.empty:
    for col in ['auc_mean', 'auc_std']:
        summary_display[col] = summary_display[col].map(lambda x: f'{float(x):.6f}')
    for col in ['fpr_at_tpr_30_mean', 'fpr_at_tpr_30_std', 'fpr_at_tpr_50_mean', 'fpr_at_tpr_50_std']:
        summary_display[col] = summary_display[col].map(lambda x: f'{float(x):.6e}')
    for col in ['gap_recovery_tpr_30_mean', 'gap_recovery_tpr_30_std', 'gap_recovery_tpr_50_mean', 'gap_recovery_tpr_50_std']:
        summary_display[col] = summary_display[col].map(lambda x: 'nan' if pd.isna(x) else f'{100.0 * float(x):.2f}%')

print('Saved table:', detail_out)
print('Saved table:', summary_out)

try:
    from IPython.display import display
    display(summary_display)
    display(detail_display)
except Exception:
    print(summary_display.to_string(index=False))
    print(detail_display.to_string(index=False))
